# 개별종목 조합B — LightGBM

`기본모델/04.LightGBM.ipynb`과 같은 `models.lightgbm.build_lightgbm_baseline`을 가져오고
조합B 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.lightgbm import build_lightgbm_baseline  # noqa: E402

MODEL_NAME = 'LightGBM'
MODEL_BUILDER = build_lightgbm_baseline


In [2]:
# 2. 조합B의 피처 값만 지정합니다.
import json

COMBINATION = 'B'
FEATURE_COLUMNS = (
    'ret_5',
    'ret_20',
    'sma_gap_5_20',
    'sma_gap_20_60',
    'rsi_14',
    'dist_high_20',
    'dist_high_60',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합B 피처: ('ret_5', 'ret_20', 'sma_gap_5_20', 'sma_gap_20_60', 'rsi_14', 'dist_high_20', 'dist_high_60')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4710,0.5012,-0.0303,0.3612,0.3734,0.0814,0.3909,0.1859,0.2921
1,2,balanced,980,20150123,20150421,0.3741,0.3978,-0.0237,0.3573,0.3594,0.0459,0.3693,0.2792,0.3314
2,3,balanced,1210,20151228,20160328,0.3594,0.3762,-0.0168,0.3587,0.3593,0.0402,0.3594,0.3598,0.3593
3,4,balanced,1439,20161202,20170228,0.4435,0.4617,-0.0183,0.3975,0.3984,0.1067,0.4111,0.2743,0.3565
4,5,balanced,1669,20171113,20180207,0.3971,0.3901,0.0070,0.3832,0.3845,0.0780,0.3853,0.3291,0.3673
5,6,balanced,1899,20181024,20190118,0.4247,0.3725,0.0522,0.4246,0.4270,0.1427,0.4325,0.4481,0.4322
6,7,balanced,2129,20190930,20191224,0.4337,0.4781,-0.0444,0.3678,0.3771,0.0799,0.3931,0.3020,0.3599
7,8,balanced,2359,20200902,20201130,0.3878,0.3476,0.0401,0.3875,0.3921,0.0894,0.3916,0.4455,0.4052
8,9,balanced,2589,20210806,20211105,0.3379,0.3914,-0.0535,0.3271,0.3428,0.0045,0.3617,0.2084,0.2774
9,10,balanced,2818,20220714,20221012,0.3803,0.3454,0.0349,0.3800,0.3868,0.0774,0.3837,0.3071,0.3522


,OOS 폴드 평균
accuracy,0.3986
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0017
macro_f1,0.3760
balanced_accuracy,0.3807
mcc,0.0750
pr_auc_macro_ovr,0.3882
down_recall,0.3289
core_harmonic_mean,0.3596


재실행 명령: python scripts/run_stock_model_experiment.py
